<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 2 — Exercises: Pydantic Validation & Gradio UIs

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Practise

Six short tasks. Fill in every `___` and run the cell.

1. **Q1** — define a Pydantic model and build a valid object
2. **Q2** — catch a bad value with `ValidationError`
3. **Q3** — see why Pydantic catches what a `TypedDict` misses
4. **Q4** — lock a field to a fixed set of values with an `Enum`
5. **Q5** — build your first `gr.Interface`
6. **Q6** — build a chatbot with `gr.ChatInterface`

> **Q1–Q5 need no API key at all.** Only the chatbot in Q6 does.

---

## 1. Setup

Run these two cells first.

In [ ]:
# Install what we need
!pip install -q openai gradio pydantic

In [ ]:
import os
from getpass import getpass
from enum import Enum
from typing import TypedDict
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
import gradio as gr

# Q1-Q5 need no key. Press Enter to skip; you can re-run this cell before Q6.
openai_api_key = getpass("OpenAI API Key (press Enter to skip - only Q6 needs it): ")

MODEL = "gpt-4o-mini"
openai_client = None
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key
    openai_client = OpenAI()
    print("Ready - key loaded, all six exercises will work.")
else:
    print("Ready - no key. Q1 to Q5 will work; re-run this cell before Q6.")

---

## 2. Pydantic — data that checks itself

A Pydantic model is a class that **describes the shape of your data and enforces it**.
Build one with bad data and it refuses, loudly, instead of letting the mistake travel
downstream.

### Q1: Define a model and build one

Write the field types, then build a valid `Student`.

In [ ]:
# Hint: name is text (str), roll is a whole number (int).
# Field(ge=..., le=...) sets a minimum and a maximum - a CGPA is out of 10.

class Student(BaseModel):
    name: ___
    roll: ___
    cgpa: float = Field(ge=0, le=___)


aditi = Student(name="Aditi", roll=42, cgpa=8.7)
print(aditi)
print("Her CGPA is:", aditi.cgpa)

### Q2: Catch a bad value

A CGPA of 99 breaks the `le=10` rule. Make it fail, and catch the error.

In [ ]:
# Hint: which exception does Pydantic raise when validation fails?
# You imported it at the top of this notebook.

try:
    rahul = Student(name="Rahul", roll=43, cgpa=___)     # out of range on purpose
    print(rahul)
except ___ as e:
    print("Rejected — and here is exactly why:\n")
    print(e)

### Q3: Pydantic vs `TypedDict`

A `TypedDict` looks like it does the same job. Put **clearly wrong** data into both
and see which one actually complains.

In [ ]:
# Hint: put text where a number belongs - e.g. roll="not-a-number", cgpa="excellent".

class StudentDict(TypedDict):
    name: str
    roll: int
    cgpa: float


# 1) The TypedDict version
sloppy = StudentDict(name="Meera", roll="not-a-number", cgpa="excellent")
print("TypedDict accepted it:", sloppy)

# 2) The Pydantic version - same bad data
try:
    Student(name="Meera", roll=___, cgpa=___)
except ValidationError as e:
    print("\nPydantic rejected it:\n")
    print(e)

### Q4: Lock a field with an `Enum`

An `Enum` makes an invalid value **impossible**, not merely unlikely.

In [ ]:
# Hint: each Enum member is name = "value". The field's type is the Enum class itself.

class Priority(str, Enum):
    urgent = "urgent"
    normal = "normal"
    low    = ___


class Ticket(BaseModel):
    customer: str
    priority: ___          # only the three values above are allowed


print(Ticket(customer="Aditi", priority="urgent"))

# Now try a fourth label - it must fail
try:
    Ticket(customer="Rahul", priority="super-urgent")
except ValidationError as e:
    print("\nRejected — the model can only ever be one of three values:\n")
    print(e)

---

## 3. Gradio — turn a function into a web app

`gr.Interface` wraps **any Python function** in a UI. In Colab it renders right below
the cell; add `share=True` inside `launch()` for a public link you can open on your phone.

### Q5: Your first Interface

No AI here — just a plain function with a UI on top. Complete the function, then
point the Interface at it.

In [ ]:
# Hint: text.split() breaks a sentence into a list of words.
#       len(...) gives you the size of a list, or the length of a string.

def count_it(text):
    words = len(text.___())        # how many words
    chars = ___(text)              # how many characters
    return f"{words} words, {chars} characters"


# Try it directly first - always cheaper to debug than a UI
print(count_it("gradio makes this easy"))

gr.Interface(
    fn=___,                        # the function to run
    inputs="textbox",
    outputs="textbox",
    title="Counter",
    flagging_mode="never",
).launch()

### Q6: A chatbot with `gr.ChatInterface`

`gr.ChatInterface` hands your function two things: the new `message`, and the whole
`history` so far. With `type="messages"`, that history is already a list of
`{"role": ..., "content": ...}` dicts — the exact shape the API wants.

**This one needs your API key** — re-run the setup cell if you skipped it.

In [ ]:
# Hint: the system message sets the personality. history already has the right
#       shape, so you can drop it straight into the list.

system_message = "___"       # give your bot a personality, e.g. "You are a pirate. Answer in pirate slang."


def chat(message, history):
    messages = [{"role": "system", "content": ___}] + history + [{"role": "user", "content": ___}]
    r = openai_client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content


gr.ChatInterface(fn=___, type="messages").launch()

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **Pydantic model** | a class that describes your data's shape — and enforces it |
| **`Field(ge=, le=)`** | a range check the model applies for you |
| **`ValidationError`** | raised the moment bad data appears, not three functions later |
| **`TypedDict`** | hints only — at runtime it is just a dict, and it accepts anything |
| **`Enum`** | an invalid value becomes impossible, not merely unlikely |
| **`gr.Interface`** | any Python function → a web UI |
| **`gr.ChatInterface`** | a chatbot, where `history` gives it memory for free |

**Finished early?** Try these:
1. Add an `email` field to `Student` and reject anything without an `@`.
2. Give the counter Interface a second output box that also shows the longest word.
3. Add `share=True` to `launch()` and open your chatbot on your phone.